# UTS Data Acquisition & Management
## Pariwisata Surabaya: Kunjungan Wisatawan per Destinasi dan Cuaca Bulanan

| | |
|---|---|
| Mata kuliah | Data Acquisition & Management — Informatika, Kelas E, INSTIKI |
| Dosen | Dr. I Gede Totok Suryawan |
| Sumber (a) berkas | Open Data Surabaya (CKAN) — *Banyaknya Kunjungan Wisatawan Nusantara dan Wisatawan Mancanegara menurut Tempat Wisata* (Disbudporapar Kota Surabaya), CSV |
| Sumber (b) API | Open-Meteo Historical Weather (archive) API — tanpa API key |
| Sumber (c) scraping | Wikipedia bahasa Indonesia — *Daftar kecamatan dan kelurahan di Kota Surabaya* |

**Alur:** Setup → Soal 1 (baca & identifikasi berkas) → Soal 2 (lisensi, PII, etika) → Soal 3 (API) → Soal 4 (robots.txt & scraping) → Soal 5 (merge) → Ringkasan.

Semua data mentah disimpan di `data/raw/` dan **tidak pernah diubah**: jika berkas mentah sudah ada, notebook tidak mengunduh ulang, melainkan memakai salinan yang ada dan memverifikasi hash MD5-nya di akhir. Hasil olahan disimpan di `data/processed/`.

## 0. Setup

> **Catatan:** Email kampus tidak tersedia; User-Agent memakai email pribadi sebagai kontak.

Email kontak diambil dari **environment variable `CONTACT_EMAIL`**, atau jika variabel itu tidak ada, dari berkas `.env` (dibaca dengan `python-dotenv`). Dengan begitu alamatnya tidak pernah tertulis di kode notebook. Nilainya juga **tidak pernah dicetak**: output hanya menampilkan struktur User-Agent dengan bagian kontak disamarkan. Berkas `.env` masuk `.gitignore`; yang di-commit hanya `.env.example`.

Fungsi bantu yang didefinisikan di bagian ini:
- `etis_get()` — request HTTP yang etis: `timeout`, `User-Agent` berisi kontak, jeda antar-request per host (`time.sleep`), retry dengan *exponential backoff* untuk status 429/5xx (menghormati header `Retry-After`), serta log status code.
- `md5_file()` — menghitung hash MD5 berkas.
- `catat_provenance()` — mencatat URL, tanggal akses, lisensi, MD5, dan metode akuisisi setiap sumber.

In [ ]:
# Import pustaka standar dan pihak ketiga yang dipakai di seluruh notebook
import hashlib
import io
import json
import os
import platform
import time
import urllib.robotparser
from datetime import datetime, date, timezone, timedelta
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Tampilkan semua kolom agar bukti di output tidak terpotong
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Konfigurasi path proyek: semua path relatif terhadap folder notebook
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Zona waktu WIB untuk mencatat tanggal akses dan tanggal proses
WIB = timezone(timedelta(hours=7))

print("Python  :", platform.python_version())
print("pandas  :", pd.__version__, "| requests:", requests.__version__)
print("RAW_DIR :", RAW_DIR.relative_to(BASE_DIR))
print("PROC_DIR:", PROCESSED_DIR.relative_to(BASE_DIR))

In [ ]:
# Ambil CONTACT_EMAIL: environment variable lebih dulu, lalu .env (load_dotenv tidak menimpa env yang sudah ada)
SUMBER_KONTAK = "environment variable" if os.environ.get("CONTACT_EMAIL", "").strip() else ".env"
load_dotenv(BASE_DIR / ".env", override=False)
CONTACT_EMAIL = os.getenv("CONTACT_EMAIL", "").strip()

# Hentikan notebook dengan pesan jelas jika kontak kosong atau masih berisi contoh
if not CONTACT_EMAIL or "@" not in CONTACT_EMAIL or CONTACT_EMAIL.endswith("example.com"):
    raise SystemExit("CONTACT_EMAIL kosong. Buat .env dari .env.example dan isi CONTACT_EMAIL, "
                     "atau set environment variable CONTACT_EMAIL.")

# User-Agent jujur: nama klien, tujuan akademik, dan kontak yang bisa dihubungi pemilik situs
UA_TEMPLAT = ("UTS-DAM-INSTIKI/1.0 (tugas kuliah Data Acquisition & Management; "
              "kontak: {kontak}) python-requests/" + requests.__version__)
USER_AGENT = UA_TEMPLAT.format(kontak=CONTACT_EMAIL)

# Bukti: tampilkan User-Agent yang dipakai, dengan nilai email disamarkan (tidak pernah dicetak)
print("Sumber CONTACT_EMAIL :", SUMBER_KONTAK)
print("User-Agent yang dipakai (kontak disamarkan):")
print(UA_TEMPLAT.format(kontak=f"<CONTACT_EMAIL, {len(CONTACT_EMAIL)} karakter, disamarkan>"))
print("Kontak terisi dan berformat email:", "@" in CONTACT_EMAIL)

In [ ]:
# Parameter etika request: timeout (connect, read), jeda minimum per host, dan batas retry
TIMEOUT = (10, 60)          # detik: 10 untuk koneksi, 60 untuk membaca respons
JEDA_DEFAULT = 2.0          # detik jeda minimum antar-request ke host yang sama
JEDA_PER_HOST = {}          # diisi dari Crawl-delay robots.txt bila ada (mis. CKAN Surabaya)
MAKS_RETRY = 4              # jumlah percobaan ulang untuk 429/5xx
STATUS_RETRY = {429, 500, 502, 503, 504}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

LOG_REQUEST = []            # log semua request: waktu, host, URL, status, durasi, percobaan
_terakhir_per_host = {}     # waktu request terakhir per host untuk menegakkan jeda


def etis_get(url, params=None, jeda=None):
    """GET yang etis: timeout, UA berkontak, jeda per host, retry + exponential backoff, log status."""
    host = urlparse(url).netloc
    jeda_min = jeda if jeda is not None else JEDA_PER_HOST.get(host, JEDA_DEFAULT)
    for percobaan in range(1, MAKS_RETRY + 2):
        # Tegakkan jeda minimum sejak request terakhir ke host yang sama
        sejak = time.monotonic() - _terakhir_per_host.get(host, -1e9)
        if sejak < jeda_min:
            time.sleep(jeda_min - sejak)
        mulai = time.monotonic()
        try:
            resp = SESSION.get(url, params=params, timeout=TIMEOUT)
            status = resp.status_code
        except requests.RequestException as err:
            resp, status = None, f"ERROR {type(err).__name__}"
        _terakhir_per_host[host] = time.monotonic()
        LOG_REQUEST.append({
            "waktu": datetime.now(WIB).strftime("%Y-%m-%d %H:%M:%S%z"),
            "host": host,
            "url": (resp.url if resp is not None else url)[:120],
            "status": status,
            "durasi_s": round(time.monotonic() - mulai, 2),
            "percobaan": percobaan,
        })
        print(f"[{status}] percobaan {percobaan} {host}{urlparse(url).path[:70]}")
        # Sukses atau error klien selain 429: kembalikan/hentikan tanpa retry
        if resp is not None and status not in STATUS_RETRY:
            resp.raise_for_status()
            return resp
        if percobaan > MAKS_RETRY:
            break
        # Exponential backoff: 2, 4, 8, 16 detik; pakai Retry-After jika server memintanya
        tunggu = 2 ** percobaan
        if resp is not None and resp.headers.get("Retry-After", "").isdigit():
            tunggu = max(tunggu, int(resp.headers["Retry-After"]))
        print(f"    -> status {status}, menunggu {tunggu} detik sebelum mencoba lagi")
        time.sleep(tunggu)
    pesan = resp.text[:200] if resp is not None else ""
    raise RuntimeError(f"Gagal mengambil {url} setelah {MAKS_RETRY + 1} percobaan "
                       f"(status terakhir {status}). {pesan}")


def md5_file(path, ukuran_blok=65536):
    """Hitung hash MD5 sebuah berkas secara bertahap (hemat memori)."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(ukuran_blok), b""):
            h.update(blok)
    return h.hexdigest()


def simpan_mentah(path, isi_bytes):
    """Simpan data mentah SEKALI saja; berkas yang sudah ada tidak boleh ditimpa."""
    if path.exists():
        raise FileExistsError(f"{path} sudah ada — data mentah tidak boleh ditimpa")
    path.write_bytes(isi_bytes)


PROVENANCE = []


def catat_provenance(sumber_id, jenis, url, halaman, lisensi, path, tanggal_akses, metode, catatan=""):
    """Catat satu baris provenance: URL, tanggal akses, lisensi, MD5, ukuran, dan metode."""
    baris = {
        "sumber_id": sumber_id,
        "jenis": jenis,
        "url": url,
        "url_halaman": halaman,
        "tanggal_akses": tanggal_akses,
        "lisensi": lisensi,
        "path_lokal": str(Path(path).relative_to(BASE_DIR)),
        "ukuran_byte": Path(path).stat().st_size,
        "md5": md5_file(path),
        "metode": metode,
        "catatan": catatan,
    }
    # Ganti entri lama dengan sumber_id yang sama agar sel bisa dijalankan ulang tanpa duplikasi
    PROVENANCE[:] = [p for p in PROVENANCE if p["sumber_id"] != sumber_id] + [baris]
    return baris


def ambil_atau_pakai(path, url, params=None, jeda=None):
    """Unduh ke data/raw jika belum ada; jika sudah ada, pakai salinan lama (raw tidak diubah).
    Mengembalikan (bytes, tanggal_akses). Tanggal akses berkas lama dibaca dari berkas .akses."""
    catatan_akses = path.with_name(path.name + ".akses")
    if path.exists():
        print(f"Berkas mentah sudah ada, tidak diunduh ulang: {path.relative_to(BASE_DIR)}")
        return path.read_bytes(), catatan_akses.read_text(encoding="utf-8").strip()
    resp = etis_get(url, params=params, jeda=jeda)
    tanggal = datetime.now(WIB).strftime("%Y-%m-%d %H:%M:%S%z")
    simpan_mentah(path, resp.content)
    catatan_akses.write_text(tanggal + "\n", encoding="utf-8")
    print(f"Disimpan: {path.relative_to(BASE_DIR)} ({len(resp.content):,} byte)")
    return resp.content, tanggal


def baca_robots(url_robots, path_simpan, jeda=None):
    """Ambil robots.txt dengan etis_get, simpan mentah, lalu parse dengan urllib.robotparser."""
    isi, tanggal = ambil_atau_pakai(path_simpan, url_robots, jeda=jeda)
    teks = isi.decode("utf-8-sig", errors="replace")  # utf-8-sig membuang BOM di awal berkas (ada di robots.txt Wikipedia)
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(url_robots)
    rp.parse(teks.splitlines())
    return rp, teks, tanggal


print("Fungsi siap: etis_get, md5_file, simpan_mentah, catat_provenance, ambil_atau_pakai, baca_robots")

## 1. Soal 1 — Membaca Berkas dan Identifikasi Data (bobot 20%)

### 1.1 Kepatuhan robots.txt portal dan unduh berkas
Sebelum mengunduh, `robots.txt` portal Open Data Surabaya (`ckan.surabaya.go.id`) diperiksa. Portal ini melarang `/api/` dan menetapkan `Crawl-delay: 10`; karena itu notebook **tidak memakai CKAN API**, hanya halaman dataset (HTML) dan URL unduhan resource, dengan jeda mengikuti `Crawl-delay`.

In [ ]:
# Identitas sumber (a): halaman dataset, halaman resource, dan URL unduhan berkas CSV
CKAN_HOST = "https://ckan.surabaya.go.id"
CKAN_DATASET = (CKAN_HOST + "/dataset/banyaknya-kunjungan-wisatawan-nusantara-dan-"
                "wisatawan-mancanegara-menurut-tempat-wisata-tahun-2025")
CKAN_RESOURCE_PAGE = CKAN_DATASET + "/resource/3a969972-e959-462d-b37b-cbdd14830f0b"
CKAN_CSV_URL = (CKAN_HOST + "/dataset/5c1cb1a5-8b39-42df-a8c3-48fff07fec6e/resource/"
                "3a969972-e959-462d-b37b-cbdd14830f0b/download/banyaknya-wisman-wisnus-2026.csv")

# Ambil dan parse robots.txt portal CKAN (disimpan mentah sebagai bukti)
rp_ckan, teks_robots_ckan, akses_robots_ckan = baca_robots(
    CKAN_HOST + "/robots.txt", RAW_DIR / "ckan_surabaya_robots.txt")
print("\n--- Isi robots.txt ckan.surabaya.go.id ---")
print(teks_robots_ckan.strip()[-400:])

# Terapkan Crawl-delay dari robots.txt sebagai jeda minimum ke host CKAN
crawl_delay = rp_ckan.crawl_delay(USER_AGENT) or JEDA_DEFAULT
JEDA_PER_HOST["ckan.surabaya.go.id"] = float(crawl_delay)
print("\nCrawl-delay diterapkan:", crawl_delay, "detik")

# Bukti can_fetch() untuk setiap URL yang akan dan tidak akan diakses
uji_ckan = {
    "halaman dataset": CKAN_DATASET,
    "halaman resource": CKAN_RESOURCE_PAGE,
    "unduhan CSV": CKAN_CSV_URL,
    "CKAN API (tidak dipakai)": CKAN_HOST + "/api/3/action/package_show",
}
pd.DataFrame([{"target": k, "url": v[:90], "can_fetch": rp_ckan.can_fetch(USER_AGENT, v)}
              for k, v in uji_ckan.items()])

In [ ]:
# Pastikan URL yang dipakai diizinkan robots.txt sebelum mengunduh
for nama in ["halaman dataset", "halaman resource", "unduhan CSV"]:
    assert rp_ckan.can_fetch(USER_AGENT, uji_ckan[nama]), f"Ditolak robots.txt: {nama}"

# Unduh berkas CSV ke data/raw (atau pakai salinan yang sudah ada tanpa mengubahnya)
PATH_CSV = RAW_DIR / "ckan_surabaya_wisman_wisnus_tempat_wisata.csv"
_, akses_csv = ambil_atau_pakai(PATH_CSV, CKAN_CSV_URL)

# Simpan halaman dataset (HTML) sebagai bukti metadata dan lisensi untuk Soal 2
PATH_CKAN_HTML = RAW_DIR / "ckan_surabaya_dataset_page.html"
_, akses_ckan_html = ambil_atau_pakai(PATH_CKAN_HTML, CKAN_DATASET)

### 1.2 Pemeriksaan isi mentah sebelum memilih parameter `read_csv`
Parameter baca dipilih **setelah** melihat isi berkas: baris pertama, karakter pemisah, *encoding*, akhir baris, dan apakah ada pemisah ribuan atau sel kosong.

In [ ]:
# Periksa byte mentah: BOM, encoding, akhir baris, dan 3 baris pertama
mentah = PATH_CSV.read_bytes()
print("Ukuran berkas      :", len(mentah), "byte")
print("Diawali BOM UTF-8  :", mentah.startswith(b"\xef\xbb\xbf"))
print("Semua byte ASCII   :", all(b < 128 for b in mentah))
print("Akhir baris CRLF   :", b"\r\n" in mentah)
print("Jumlah ';' baris 1 :", mentah.splitlines()[0].count(b";"),
      "| jumlah ',' baris 1:", mentah.splitlines()[0].count(b","))
print("\n3 baris pertama:")
for baris in mentah.decode("utf-8").splitlines()[:3]:
    print(" ", baris)

**Parameter `pd.read_csv` yang dipakai dan alasannya**

| Parameter | Nilai | Alasan (berdasarkan pemeriksaan di atas) |
|---|---|---|
| `sep` | `";"` | Kolom dipisahkan titik koma (baris header berisi 8 `;` dan 0 `,`). |
| `encoding` | `"utf-8"` | Semua byte ASCII dan tanpa BOM, jadi `utf-8` aman; ditulis eksplisit agar hasil sama di Windows/Mac/Linux (default Windows bisa `cp1252`). |
| `dtype` | `kode_propinsi`, `kode_kota` → `str`; `wisman`, `wisnus`, `tahun` → `int64` | Kode wilayah adalah label, bukan angka untuk dihitung (dan aman dari hilangnya nol di depan). Jumlah kunjungan dan tahun adalah bilangan bulat. Menetapkan `int64` juga berfungsi sebagai validasi: baca akan gagal jika ada nilai non-angka. |
| `na_values` | tidak dipakai | Tidak ada sel kosong maupun penanda seperti `-`/`NA` (dibuktikan di bawah). Nilai `0` adalah jumlah kunjungan nol yang sah, **bukan** data hilang, sehingga tidak boleh dijadikan NaN. |
| `thousands` | tidak dipakai | Angka ditulis tanpa pemisah ribuan (tidak ada `.` atau `,` di kolom angka). |
| `skiprows` / `sheet_name` | tidak dipakai | Baris pertama langsung header; berkas berformat CSV, bukan Excel. |

In [ ]:
# Baca berkas dengan parameter yang sudah dijustifikasi di tabel atas
DTYPE_KUNJUNGAN = {
    "kode_propinsi": "str", "nama_propinsi": "str", "kode_kota": "str", "nama_kota": "str",
    "nama_destinasi": "str", "wisman": "int64", "wisnus": "int64", "periode": "str", "tahun": "int64",
}
df_kunjungan_raw = pd.read_csv(PATH_CSV, sep=";", encoding="utf-8", dtype=DTYPE_KUNJUNGAN)

print("Bentuk (baris, kolom):", df_kunjungan_raw.shape)
print("Syarat minimal 100 baris terpenuhi:", len(df_kunjungan_raw) >= 100)
df_kunjungan_raw.head()

In [ ]:
# Bukti: tidak ada nilai kosong, dan tidak ada pemisah ribuan pada kolom angka di teks mentah
teks_mentah = pd.read_csv(PATH_CSV, sep=";", encoding="utf-8", dtype=str, keep_default_na=False)
print("Sel kosong per kolom:", (teks_mentah == "").sum().to_dict())
print("NaN setelah dibaca  :", int(df_kunjungan_raw.isna().sum().sum()))
print("Ada '.' atau ',' di kolom angka:",
      bool(teks_mentah[["wisman", "wisnus"]].apply(lambda s: s.str.contains(r"[.,]")).any().any()))

### 1.3 Identifikasi data

In [ ]:
# Struktur: Wisnus dan Wisman berada di KOLOM terpisah (format lebar), satu baris = destinasi x bulan
print("Nama kolom persis:", list(df_kunjungan_raw.columns))
print("Format kolom periode: nama bulan bahasa Indonesia ->", df_kunjungan_raw["periode"].unique().tolist())

# Nama destinasi: ada yang berbeda hanya karena spasi di ujung
nama_asli = df_kunjungan_raw["nama_destinasi"]
print("\nNama destinasi unik (apa adanya)     :", nama_asli.nunique())
print("Nama destinasi unik (setelah strip())  :", nama_asli.str.strip().nunique())
print("Nama yang mengandung spasi di ujung   :", sorted(nama_asli[nama_asli != nama_asli.str.strip()].map(repr).unique()))

# Cakupan waktu: tabulasi silang bulan x tahun
URUTAN_BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
                "Agustus", "September", "Oktober", "November", "Desember"]
pd.crosstab(pd.Categorical(df_kunjungan_raw["periode"], URUTAN_BULAN, ordered=True),
            df_kunjungan_raw["tahun"], rownames=["periode"], margins=True)

In [ ]:
# Tipe data tiap kolom beserta satuan dan keterangan
satuan = {
    "kode_propinsi": "kode wilayah BPS (label)", "nama_propinsi": "teks",
    "kode_kota": "kode wilayah BPS (label)", "nama_kota": "teks",
    "nama_destinasi": "teks (nama tempat wisata)",
    "wisman": "orang (kunjungan wisatawan mancanegara per bulan)",
    "wisnus": "orang (kunjungan wisatawan nusantara per bulan)",
    "periode": "nama bulan (bahasa Indonesia)", "tahun": "tahun (YYYY)",
}
identifikasi = pd.DataFrame({
    "dtype": df_kunjungan_raw.dtypes.astype(str),
    "nilai_unik": df_kunjungan_raw.nunique(),
    "jumlah_nol": [(df_kunjungan_raw[c] == 0).sum() if df_kunjungan_raw[c].dtype.kind in "iu" else "-"
                   for c in df_kunjungan_raw.columns],
    "nilai_kosong": df_kunjungan_raw.isna().sum(),
    "satuan": pd.Series(satuan),
})
identifikasi

In [ ]:
# Ringkasan identifikasi data untuk dilaporkan
ukuran_byte = PATH_CSV.stat().st_size
ringkasan_identifikasi = {
    "Jenis data": "Terstruktur (tabel CSV, skema kolom tetap)",
    "Jumlah baris x kolom": f"{df_kunjungan_raw.shape[0]} x {df_kunjungan_raw.shape[1]}",
    "Cakupan wilayah": f"{df_kunjungan_raw['nama_kota'].iloc[0].title()}, "
                       f"{df_kunjungan_raw['nama_propinsi'].iloc[0].title()} — "
                       f"{df_kunjungan_raw['nama_destinasi'].str.strip().nunique()} destinasi wisata",
    "Cakupan waktu": f"{df_kunjungan_raw.loc[0, 'periode']} {df_kunjungan_raw['tahun'].min()} s.d. "
                     f"{df_kunjungan_raw['periode'].iloc[-1]} {df_kunjungan_raw['tahun'].max()} (bulanan)",
    "Ukuran berkas": f"{ukuran_byte:,} byte ({ukuran_byte / 1024:.2f} KiB)",
    "Nilai kosong": int(df_kunjungan_raw.isna().sum().sum()),
    "Nilai nol wisman / wisnus": f"{(df_kunjungan_raw['wisman'] == 0).sum()} / {(df_kunjungan_raw['wisnus'] == 0).sum()}",
}
pd.Series(ringkasan_identifikasi, name="nilai").to_frame()

### 1.4 Tahun analisis (diambil dari data, bukan asumsi)
Kolom `tahun` ternyata berisi **dua** tahun. `TAHUN_DATA` ditetapkan sebagai tahun yang datanya **lengkap 12 bulan** di kolom `tahun`; tahun yang belum lengkap tidak dipakai di analisis, tetapi tetap utuh di `data/raw/`.

In [ ]:
# Hitung jumlah bulan per tahun langsung dari kolom tahun dan periode
bulan_per_tahun = df_kunjungan_raw.groupby("tahun")["periode"].nunique()
print("Jumlah bulan unik per tahun:", bulan_per_tahun.to_dict())

tahun_lengkap = bulan_per_tahun[bulan_per_tahun == 12].index.tolist()
assert len(tahun_lengkap) == 1, f"Diharapkan tepat satu tahun lengkap, ditemukan {tahun_lengkap}"
TAHUN_DATA = int(tahun_lengkap[0])
print("TAHUN_DATA (tahun lengkap 12 bulan dari kolom 'tahun'):", TAHUN_DATA)

# Subset analisis: hanya baris TAHUN_DATA (data mentah tidak diubah)
df_kunjungan = df_kunjungan_raw[df_kunjungan_raw["tahun"] == TAHUN_DATA].copy()
BULAN_TERCAKUP = [b for b in URUTAN_BULAN if b in set(df_kunjungan["periode"])]
print("Baris untuk analisis:", len(df_kunjungan), "| bulan tercakup:", BULAN_TERCAKUP)
print("Baris tahun lain yang dikecualikan:", len(df_kunjungan_raw) - len(df_kunjungan))
print(f"Nilai nol di subset {TAHUN_DATA}: wisman {(df_kunjungan['wisman'] == 0).sum()}, "
      f"wisnus {(df_kunjungan['wisnus'] == 0).sum()} (dari {len(df_kunjungan)} baris; nol = tidak ada kunjungan tercatat, bukan data hilang)")

### 1.5 Pemeriksaan kualitas: bulan dengan angka identik
Dua bulan yang angkanya sama persis untuk **semua** destinasi hampir mustahil terjadi secara alami, sehingga patut dicurigai sebagai kesalahan salin di sumber. Temuan dilaporkan dan **tidak dikoreksi** (data mentah tidak diubah; tidak ada dasar untuk menebak angka sebenarnya).

In [ ]:
# Bandingkan setiap pasangan bulan di TAHUN_DATA: apakah wisnus DAN wisman identik untuk semua destinasi?
pivot = (df_kunjungan.assign(nama_destinasi=df_kunjungan["nama_destinasi"].str.strip())
         .pivot(index="nama_destinasi", columns="periode", values=["wisnus", "wisman"]))
BULAN_IDENTIK = []
for i, b1 in enumerate(BULAN_TERCAKUP):
    for b2 in BULAN_TERCAKUP[i + 1:]:
        if all((pivot[k][b1] == pivot[k][b2]).all() for k in ["wisnus", "wisman"]):
            BULAN_IDENTIK.append((b1, b2))
print("Pasangan bulan yang identik untuk semua destinasi:", BULAN_IDENTIK or "tidak ada")
if BULAN_IDENTIK:
    b1, b2 = BULAN_IDENTIK[0]
    display(pivot.xs(b1, axis=1, level=1).join(pivot.xs(b2, axis=1, level=1), lsuffix=f"_{b1}", rsuffix=f"_{b2}").head())
    print("Catatan: kemungkinan kesalahan salin di sumber; dilaporkan sebagai keterbatasan, tidak dikoreksi.")

### 1.6 Estimasi kebutuhan storage 1 tahun

**Rumus:** `ukuran berkas × 12 × 1,3`

**Asumsi eksplisit:**
1. Portal memperbarui berkas ini dan setiap pembaruan diunduh sebagai **satu salinan baru per bulan** (12 salinan per tahun) — salinan lama tidak ditimpa karena data mentah bersifat *immutable*.
2. Ukuran setiap salinan **sama dengan ukuran berkas saat ini** (pertumbuhan baris diabaikan; ditutup oleh cadangan).
3. Cadangan **30%** (faktor 1,3) untuk berkas pendukung (HTML bukti lisensi, robots.txt, log), metadata, dan pertumbuhan baris.
4. Tanpa kompresi dan tanpa memperhitungkan *overhead* sistem berkas/replikasi.

In [ ]:
# Estimasi storage 1 tahun = ukuran berkas x 12 x 1,3
FREKUENSI_PER_TAHUN = 12
CADANGAN = 1.3
estimasi_byte = ukuran_byte * FREKUENSI_PER_TAHUN * CADANGAN
print(f"Ukuran berkas saat ini : {ukuran_byte:,} byte")
print(f"Estimasi 1 tahun       : {ukuran_byte:,} x {FREKUENSI_PER_TAHUN} x {CADANGAN} = {estimasi_byte:,.0f} byte")
print(f"                       = {estimasi_byte / 1024:,.2f} KiB = {estimasi_byte / 1024**2:,.3f} MiB")

In [ ]:
# Catat provenance sumber (a): berkas CSV dan halaman dataset sebagai bukti metadata
METODE_SKRIP = "diunduh lewat skrip Python (requests) dengan User-Agent jujur berisi kontak; jeda sesuai Crawl-delay robots.txt"
catat_provenance("A_csv_kunjungan", "berkas (CSV)", CKAN_CSV_URL, CKAN_RESOURCE_PAGE,
                 "tidak dicantumkan di metadata CKAN (\"No License Provided\")", PATH_CSV, akses_csv, METODE_SKRIP,
                 "Disbudporapar Kota Surabaya; berisi Jan 2025 s.d. Okt 2026")
catat_provenance("A_html_dataset", "bukti (HTML halaman dataset)", CKAN_DATASET, CKAN_DATASET,
                 "tidak dicantumkan di metadata CKAN", PATH_CKAN_HTML, akses_ckan_html, METODE_SKRIP,
                 "bukti metadata & lisensi untuk Soal 2")
catat_provenance("A_robots", "bukti (robots.txt)", CKAN_HOST + "/robots.txt", CKAN_HOST + "/robots.txt",
                 "-", RAW_DIR / "ckan_surabaya_robots.txt", akses_robots_ckan, METODE_SKRIP)
pd.DataFrame(PROVENANCE)[["sumber_id", "tanggal_akses", "lisensi", "ukuran_byte", "md5"]]

## 2. Soal 2 — Lisensi, Ketentuan Portal, PII, dan Data Ethics Checklist

### 2.1 Lisensi dataset dan ketentuan portal
Bukti diambil dari HTML halaman dataset dan halaman *About* portal yang disimpan di `data/raw/`.

In [ ]:
# Ekstrak bagian "License" dari HTML halaman dataset CKAN sebagai bukti lisensi
sup_ckan = BeautifulSoup(PATH_CKAN_HTML.read_text(encoding="utf-8"), "lxml")
blok_lisensi = sup_ckan.select_one("section.module-narrow.license, .license")
teks_lisensi = " ".join(blok_lisensi.get_text(" ", strip=True).split()) if blok_lisensi else "(blok lisensi tidak ditemukan)"
print("URL halaman   :", CKAN_DATASET)
print("Tanggal akses :", akses_ckan_html)
print("Kutipan blok lisensi:", teks_lisensi)

# Ambil halaman About portal (ketentuan/dasar hukum portal) sebagai bukti
CKAN_ABOUT = CKAN_HOST + "/about"
assert rp_ckan.can_fetch(USER_AGENT, CKAN_ABOUT)
PATH_ABOUT = RAW_DIR / "ckan_surabaya_about.html"
_, akses_about = ambil_atau_pakai(PATH_ABOUT, CKAN_ABOUT)
sup_about = BeautifulSoup(PATH_ABOUT.read_text(encoding="utf-8"), "lxml")
teks_about = " ".join((sup_about.select_one("main, article, .module-content") or sup_about).get_text(" ", strip=True).split())
print("\nURL halaman ketentuan portal:", CKAN_ABOUT, "| tanggal akses:", akses_about)
for kunci in ["Peraturan Presiden Nomor 39 Tahun 2019", "Peraturan Walikota Surabaya Nomor 126 Tahun 2021",
              "dapat diakses oleh masyarakat secara terbuka"]:
    i = teks_about.find(kunci)
    print(f"- Kutipan: \"...{teks_about[max(0, i - 60):i + len(kunci) + 60]}...\"" if i >= 0 else f"- (tidak ditemukan: {kunci})")

catat_provenance("A_html_about", "bukti (HTML ketentuan portal)", CKAN_ABOUT, CKAN_ABOUT, "-",
                 PATH_ABOUT, akses_about, METODE_SKRIP, "dasar hukum & pernyataan keterbukaan portal")

**Kesimpulan lisensi:** lisensi **tidak dicantumkan di metadata CKAN** (halaman dataset menampilkan *"No License Provided"*). Portal tidak menyediakan halaman *Terms of Service* tersendiri. Halaman *About* menyatakan portal diselenggarakan berdasarkan Perpres 39/2019 dan Perwali Surabaya 126/2021, dan bahwa data *"dapat diakses oleh masyarakat secara terbuka"* untuk dicari, dilihat, dan diunduh. Karena tidak ada lisensi eksplisit, penggunaan dibatasi untuk keperluan akademik nonkomersial dengan atribusi penuh kepada Disbudporapar Kota Surabaya/Open Data Surabaya, dan data tidak diredistribusikan dengan klaim lisensi tertentu.

### 2.2 Identifikasi PII dan quasi-identifier

In [ ]:
# Penilaian PII / quasi-identifier per kolom (dataset berisi agregat kunjungan per destinasi per bulan)
penilaian_pii = pd.DataFrame([
    ("kode_propinsi", "Tidak", "Tidak", "Kode wilayah provinsi; sama untuk semua baris."),
    ("nama_propinsi", "Tidak", "Tidak", "Nama provinsi; bukan atribut individu."),
    ("kode_kota", "Tidak", "Tidak", "Kode wilayah kota; sama untuk semua baris."),
    ("nama_kota", "Tidak", "Tidak", "Nama kota; bukan atribut individu."),
    ("nama_destinasi", "Tidak", "Tidak", "Nama tempat wisata publik (fasilitas), bukan orang."),
    ("wisman", "Tidak", "Tidak", "Jumlah agregat kunjungan bulanan; tidak memuat atribut individu."),
    ("wisnus", "Tidak", "Tidak", "Jumlah agregat kunjungan bulanan; tidak memuat atribut individu."),
    ("periode", "Tidak", "Tidak", "Bulan pelaporan (granular bulanan, bukan tanggal kunjungan individu)."),
    ("tahun", "Tidak", "Tidak", "Tahun pelaporan."),
], columns=["kolom", "PII", "quasi_identifier", "alasan"])
assert set(penilaian_pii["kolom"]) == set(df_kunjungan_raw.columns)
penilaian_pii

In [ ]:
# Cek risiko sel kecil: nilai agregat sangat kecil (1-4) secara teori bisa menyempitkan identitas
sel_kecil = df_kunjungan_raw[(df_kunjungan_raw["wisman"].between(1, 4))]
print("Baris dengan wisman 1-4 orang:", len(sel_kecil), "dari", len(df_kunjungan_raw))
print("Tanpa atribut lain (nama, asal, umur, tanggal), angka ini tidak dapat ditautkan ke individu.")

**Kesimpulan PII:** **tidak ada** kolom PII maupun quasi-identifier. Semua kolom berupa wilayah administratif, nama fasilitas publik, periode bulanan, atau jumlah agregat. Ada beberapa sel dengan jumlah wisman kecil (1–4), tetapi tanpa atribut demografis atau tanggal kunjungan sel tersebut tidak dapat ditautkan ke individu. Data cuaca (Soal 3) dan tabel kecamatan Wikipedia (Soal 4) juga tidak memuat data pribadi.

### 2.3 Kontak di User-Agent: email kampus vs email pribadi
Soal meminta **email kampus** dicantumkan di User-Agent agar pemilik situs dapat menghubungi pengakses. Mahasiswa **tidak diberi email kampus**, sehingga User-Agent memakai **email pribadi** mahasiswa sebagai kontak. Alamat itu dibaca dari environment variable `CONTACT_EMAIL` atau berkas `.env` lokal (lihat sel Setup), tidak ditulis di notebook, tidak dicetak di output, dan tidak di-commit. Fungsinya sama dengan email kampus: memberi pemilik situs jalur untuk menghubungi pengakses.

### 2.4 Data Ethics Checklist (9 butir)

In [ ]:
# Data Ethics Checklist 9 butir: butir, status, bukti (merujuk output/berkas di notebook ini)
checklist = pd.DataFrame([
    (1, "Sumber & kewenangan penerbit", "Terpenuhi",
     "Penerbit: Dinas Kebudayaan, Kepemudaan dan Olah Raga serta Pariwisata (Disbudporapar) Kota Surabaya, "
     "melalui portal resmi Open Data Surabaya (ckan.surabaya.go.id; Perwali 126/2021). Cuaca: Open-Meteo. "
     "Tabel kecamatan: Wikipedia ID (kode Kemendagri). Bukti: Soal 1.1, 2.1, data/raw/ckan_surabaya_dataset_page.html."),
    (2, "Lisensi", "Terpenuhi dengan catatan",
     "Dataset kunjungan: lisensi tidak dicantumkan di metadata CKAN (\"No License Provided\"), dipakai terbatas untuk "
     "akademik dengan atribusi. Open-Meteo: data CC BY 4.0. Wikipedia: CC BY-SA 4.0. Bukti: Soal 2.1, 3.1, 4.3."),
    (3, "Ketentuan layanan portal", "Terpenuhi",
     "Halaman About Open Data Surabaya (data terbuka untuk dicari, dilihat, diunduh); ketentuan Open-Meteo "
     "(gratis nonkomersial, <10.000 panggilan/hari); kebijakan User-Agent Wikimedia. Bukti: Soal 2.1, 3.1, 4.1."),
    (4, "PII / quasi-identifier", "Terpenuhi",
     "Tidak ada PII maupun quasi-identifier; data agregat bulanan per destinasi. Bukti: tabel penilaian per kolom Soal 2.2."),
    (5, "Minimisasi data", "Terpenuhi",
     "Hanya satu resource CSV, satu titik koordinat cuaca, variabel cuaca harian seperlunya, dan satu halaman Wikipedia. "
     "Analisis hanya memakai TAHUN_DATA; baris tahun lain tetap di raw tetapi tidak diproses. Dataset kandidat lain tidak disimpan."),
    (6, "robots.txt & rate limit", "Terpenuhi",
     "robots.txt CKAN & Wikipedia dibaca dengan urllib.robotparser dan can_fetch() ditampilkan; CKAN API (/api/) "
     "dilarang sehingga tidak dipakai; Crawl-delay 10 detik ditegakkan; etis_get() memakai timeout, jeda, dan backoff "
     "untuk 429/5xx. Bukti: Soal 1.1, 4.1, log request di Ringkasan."),
    (7, "Keamanan kredensial", "Terpenuhi",
     "Tidak ada API key/token. Email kontak dibaca dari environment variable atau .env (python-dotenv), tidak pernah "
     "dicetak (output menyamarkannya); .env ada di .gitignore; hanya .env.example yang di-commit; "
     "scripts/verify_submission.py memastikan nilainya tidak ada di repo maupun output. Bukti: sel Setup, Soal 2.3, .gitignore."),
    (8, "Atribusi / sitasi", "Terpenuhi",
     "Atribusi ditulis di README, gabungan_metadata.json, dan Ringkasan: Disbudporapar/Open Data Surabaya, "
     "Open-Meteo (CC BY 4.0), Wikipedia kontributor (CC BY-SA 4.0)."),
    (9, "Transparansi & reproducibility", "Terpenuhi",
     "Satu notebook end-to-end; provenance.csv berisi URL, tanggal akses, lisensi, MD5; data mentah immutable dan "
     "MD5 diverifikasi ulang; requirements.txt; gabungan_metadata.json mencatat parameter merge."),
], columns=["no", "butir", "status", "bukti"])

# Simpan checklist sebagai markdown di root proyek
with open(BASE_DIR / "data_ethics_checklist.md", "w", encoding="utf-8") as f:
    f.write("# Data Ethics Checklist — UTS DAM Pariwisata Surabaya\n\n")
    f.write("Dihasilkan otomatis oleh `uts_dam.ipynb` (Soal 2.4). Kolom *bukti* merujuk ke bagian notebook dan berkas di repo.\n\n")
    f.write("| No | Butir | Status | Bukti |\n|---|---|---|---|\n")
    for r in checklist.itertuples(index=False):
        f.write(f"| {r.no} | {r.butir} | {r.status} | {r.bukti.replace('|', '/')} |\n")
print("Disimpan: data_ethics_checklist.md")
with pd.option_context("display.max_colwidth", None):
    display(checklist.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## 3. Soal 3 — Akuisisi API: Open-Meteo Historical Weather

### 3.1 Pemilihan API dan desain permintaan
**Kenapa bukan API resmi portal sumber?** Soal meminta mengutamakan API resmi portal sumber. Portal Open Data Surabaya memang menyediakan CKAN Action API (`/api/3/action/...`), tetapi `robots.txt` portal menulis `Disallow: /api/` (bukti: tabel `can_fetch()` di Soal 1.1, baris *CKAN API* = `False`). Karena itu, sesuai alternatif yang disebut soal, dipakai API publik yang relevan dengan topik: **Open-Meteo** untuk cuaca di lokasi dataset (Kota Surabaya).

- **Endpoint:** `https://archive-api.open-meteo.com/v1/archive` (tanpa API key).
- **Titik:** satu titik Kota Surabaya, lat **-7.2575**, lon **112.7521** — titik pusat kota yang ditetapkan untuk tugas ini. Sebagai pembanding, infobox Wikipedia *Kota Surabaya* mencantumkan -7.2463, 112.7378 (selisih ±2 km); pada resolusi grid reanalisis Open-Meteo (±9–25 km) perbedaan ini tidak berarti. Satu permintaan untuk satu titik.
- **Rentang waktu:** disamakan dengan bulan yang tercakup di data kunjungan (`TAHUN_DATA`, bulan pertama s.d. terakhir).
- **Variabel harian:** suhu rata-rata/maks/min, curah hujan, jam hujan, durasi sinar matahari, kecepatan angin maks.
- **Ketentuan layanan:** API gratis untuk penggunaan nonkomersial dengan batas di bawah 10.000 panggilan per hari; data berlisensi **CC BY 4.0** (https://open-meteo.com/en/terms, https://open-meteo.com/en/license). Halaman lisensi dan ketentuan itu diunduh (setelah cek `robots.txt`) dan kutipannya ditampilkan di bawah sebagai bukti.
- **Penanganan status:** `etis_get()` mengulang permintaan pada 429/5xx dengan *exponential backoff* (2, 4, 8, 16 detik, atau sesuai `Retry-After`). Jika Open-Meteo membalas *"Daily API request limit exceeded"*, batas harian telah habis untuk IP tersebut, sehingga setelah retry habis notebook berhenti dengan pesan jelas (coba lagi besok).

In [ ]:
# Parameter permintaan Open-Meteo: rentang tanggal diturunkan dari bulan yang ada di data kunjungan
LAT, LON = -7.2575, 112.7521
no_bulan = {nama: i + 1 for i, nama in enumerate(URUTAN_BULAN)}
bulan_awal = min(no_bulan[b] for b in BULAN_TERCAKUP)
bulan_akhir = max(no_bulan[b] for b in BULAN_TERCAKUP)
TGL_MULAI = date(TAHUN_DATA, bulan_awal, 1)
TGL_SELESAI = (date(TAHUN_DATA + (bulan_akhir == 12), bulan_akhir % 12 + 1, 1) - timedelta(days=1))

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
VARIABEL_HARIAN = ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "precipitation_sum",
                   "precipitation_hours", "sunshine_duration", "wind_speed_10m_max"]
params_cuaca = {
    "latitude": LAT, "longitude": LON,
    "start_date": TGL_MULAI.isoformat(), "end_date": TGL_SELESAI.isoformat(),
    "daily": ",".join(VARIABEL_HARIAN), "timezone": "Asia/Jakarta",
}
print("Rentang data kunjungan :", BULAN_TERCAKUP[0], "-", BULAN_TERCAKUP[-1], TAHUN_DATA)
print("Rentang Open-Meteo     :", params_cuaca["start_date"], "s.d.", params_cuaca["end_date"])
print("URL lengkap            :", requests.Request("GET", OPEN_METEO_URL, params=params_cuaca).prepare().url)

In [ ]:
# Satu permintaan ke Open-Meteo (dengan retry/backoff); simpan respons JSON mentah apa adanya
PATH_CUACA_JSON = RAW_DIR / f"open_meteo_surabaya_harian_{TAHUN_DATA}.json"
isi_cuaca, akses_cuaca = ambil_atau_pakai(PATH_CUACA_JSON, OPEN_METEO_URL, params=params_cuaca)
data_cuaca = json.loads(isi_cuaca)

# Periksa respons: Open-Meteo mengembalikan {"error": true, "reason": ...} bila ada masalah
assert not data_cuaca.get("error"), data_cuaca.get("reason")
print("Kunci tingkat atas JSON:", list(data_cuaca.keys()))

# Bukti penanganan status code: log request ke host API ini (kosong jika JSON mentah sudah ada sebelumnya)
log_api = [r for r in LOG_REQUEST if r["host"] == "archive-api.open-meteo.com"]
display(pd.DataFrame(log_api)) if log_api else print("Tidak ada request API baru (memakai JSON mentah yang sudah ada).")

In [ ]:
# Bukti lisensi & ketentuan Open-Meteo: cek robots.txt situs, lalu simpan halaman lisensi dan ketentuan sebagai bukti
OM_HOST = "https://open-meteo.com"
rp_om, _, akses_robots_om = baca_robots(OM_HOST + "/robots.txt", RAW_DIR / "open_meteo_robots.txt")
bukti_om = {}
for nama, url in {"lisensi": OM_HOST + "/en/license", "ketentuan": OM_HOST + "/en/terms"}.items():
    assert rp_om.can_fetch(USER_AGENT, url), f"robots.txt menolak {url}"
    path = RAW_DIR / f"open_meteo_{nama}.html"
    isi, akses = ambil_atau_pakai(path, url)
    teks = " ".join(BeautifulSoup(isi, "lxml").get_text(" ", strip=True).split())
    bukti_om[nama] = (url, path, akses, teks)

# Tampilkan kutipan kalimat kunci sebagai bukti klaim lisensi (CC BY 4.0) dan batas pemakaian nonkomersial
for nama, kunci in [("lisensi", "CC BY 4.0"), ("ketentuan", "non-commercial"), ("ketentuan", "10,000")]:
    url, _, akses, teks = bukti_om[nama]
    i = teks.find(kunci)
    kutipan = f"\"...{teks[max(0, i - 90):i + len(kunci) + 90]}...\"" if i >= 0 else f"(frasa '{kunci}' tidak ditemukan — periksa halaman)"
    print(f"[{nama}] {url} (akses {akses}):\n   {kutipan}")

### 3.2 Normalisasi JSON menjadi tabel
Metadata respons (koordinat, elevasi, zona waktu) dinormalisasi dengan `pd.json_normalize`, sedangkan deret harian (`daily`, berupa *dict of lists*) dikonstruksi menjadi DataFrame satu baris per hari.

In [ ]:
# Metadata respons dengan json_normalize (kolom bersarang 'daily' dan 'daily_units' dipisah)
meta_cuaca = pd.json_normalize({k: v for k, v in data_cuaca.items() if k not in ("daily", "daily_units")})
display(meta_cuaca)

# Satuan setiap variabel harian
satuan_cuaca = pd.Series(data_cuaca["daily_units"], name="satuan")
display(satuan_cuaca.to_frame().T)

# Konstruksi tabel harian: satu baris per tanggal
df_cuaca_harian = pd.DataFrame(data_cuaca["daily"])
df_cuaca_harian["time"] = pd.to_datetime(df_cuaca_harian["time"])
df_cuaca_harian = df_cuaca_harian.rename(columns={"time": "tanggal"})
df_cuaca_harian["sunshine_duration"] = df_cuaca_harian["sunshine_duration"] / 3600  # detik -> jam

print("Jumlah baris tabel cuaca harian:", len(df_cuaca_harian), "| syarat minimal 30 baris:", len(df_cuaca_harian) >= 30)
print("Nilai kosong per kolom:", df_cuaca_harian.isna().sum().to_dict())
df_cuaca_harian.head()

In [ ]:
# Agregasi harian -> bulanan agar sejajar dengan granularitas data kunjungan
df_cuaca_bulanan = (
    df_cuaca_harian.assign(bulan=df_cuaca_harian["tanggal"].dt.month)
    .groupby("bulan")
    .agg(suhu_rata2_c=("temperature_2m_mean", "mean"),
         suhu_maks_c=("temperature_2m_max", "max"),
         suhu_min_c=("temperature_2m_min", "min"),
         curah_hujan_mm=("precipitation_sum", "sum"),
         jam_hujan=("precipitation_hours", "sum"),
         hari_hujan=("precipitation_sum", lambda s: int((s >= 1.0).sum())),
         sinar_matahari_jam=("sunshine_duration", "sum"),
         angin_maks_kmh=("wind_speed_10m_max", "max"),
         jumlah_hari=("tanggal", "count"))
    .round(2)
    .reset_index()
)
print("Tabel cuaca bulanan:", df_cuaca_bulanan.shape)
df_cuaca_bulanan

In [ ]:
# Simpan hasil normalisasi ke data/processed dan catat provenance sumber (b)
df_cuaca_harian.to_csv(PROCESSED_DIR / "cuaca_harian_surabaya.csv", index=False)
df_cuaca_bulanan.to_csv(PROCESSED_DIR / "cuaca_bulanan_surabaya.csv", index=False)
catat_provenance("B_open_meteo", "API (JSON)",
                 requests.Request("GET", OPEN_METEO_URL, params=params_cuaca).prepare().url,
                 "https://open-meteo.com/en/docs/historical-weather-api",
                 "CC BY 4.0 (https://open-meteo.com/en/license)", PATH_CUACA_JSON, akses_cuaca,
                 "satu request API lewat etis_get (UA jujur, timeout, retry/backoff 429/5xx)",
                 f"titik Kota Surabaya lat {LAT}, lon {LON}; harian {TGL_MULAI} s.d. {TGL_SELESAI}")
for nama, (url, path, akses, _) in bukti_om.items():
    catat_provenance(f"B_html_{nama}", f"bukti (HTML {nama} Open-Meteo)", url, url, "-", path, akses,
                     "diunduh lewat etis_get setelah can_fetch()=True")
catat_provenance("B_robots", "bukti (robots.txt)", OM_HOST + "/robots.txt", OM_HOST + "/robots.txt", "-",
                 RAW_DIR / "open_meteo_robots.txt", akses_robots_om, "diunduh lewat etis_get")
print("Disimpan: data/processed/cuaca_harian_surabaya.csv, data/processed/cuaca_bulanan_surabaya.csv")

## 4. Soal 4 — robots.txt dan Web Scraping Wikipedia

### 4.1 Pemeriksaan robots.txt
Target: https://id.wikipedia.org/wiki/Daftar_kecamatan_dan_kelurahan_di_Kota_Surabaya. Scraping hanya dilakukan jika `can_fetch()` mengizinkan User-Agent notebook ini.

In [ ]:
# Ambil robots.txt Wikipedia ID, tampilkan bagian yang relevan, lalu uji can_fetch untuk UA kita
WIKI_URL = "https://id.wikipedia.org/wiki/Daftar_kecamatan_dan_kelurahan_di_Kota_Surabaya"
rp_wiki, teks_robots_wiki, akses_robots_wiki = baca_robots(
    "https://id.wikipedia.org/robots.txt", RAW_DIR / "wikipedia_id_robots.txt")

baris_robots = teks_robots_wiki.splitlines()
print(f"robots.txt Wikipedia ID: {len(baris_robots)} baris. Cuplikan aturan untuk 'User-agent: *':")
mulai = next(i for i, b in enumerate(baris_robots) if b.strip().lower() == "user-agent: *")
print("\n".join(baris_robots[mulai:mulai + 25]))
print("...")
print("Aturan Disallow yang menyentuh '/wiki/':",
      [b for b in baris_robots[mulai:] if b.lower().startswith("disallow: /wiki/")][:8], "...")

izin_wiki = rp_wiki.can_fetch(USER_AGENT, WIKI_URL)
print("\ncan_fetch(USER_AGENT, WIKI_URL) =", izin_wiki)
print("can_fetch(USER_AGENT, '/w/index.php?action=edit') =",
      rp_wiki.can_fetch(USER_AGENT, "https://id.wikipedia.org/w/index.php?title=X&action=edit"))

### 4.2 Scraping tabel (hanya jika diizinkan)

In [ ]:
# Hentikan bila robots.txt tidak mengizinkan; jika diizinkan, ambil HTML dan simpan mentah
if not izin_wiki:
    raise SystemExit("robots.txt tidak mengizinkan halaman ini untuk UA kita — scraping dihentikan.")

PATH_WIKI_HTML = RAW_DIR / "wikipedia_kecamatan_surabaya.html"
isi_wiki, akses_wiki = ambil_atau_pakai(PATH_WIKI_HTML, WIKI_URL)

# Parse tabel HTML dengan pd.read_html; pilih tabel yang memiliki kolom 'Kecamatan'
tabel_wiki = pd.read_html(io.StringIO(isi_wiki.decode("utf-8")))
print("Jumlah tabel di halaman:", len(tabel_wiki), "| bentuk:", [t.shape for t in tabel_wiki])
df_kec_raw = next(t for t in tabel_wiki if any("Kecamatan" in str(c) for c in t.columns))
df_kec_raw.head()

In [ ]:
# Rapikan: hapus penanda catatan kaki [n] di nama kolom, buang baris total, normalisasi nama kecamatan
df_kec = df_kec_raw.copy()
df_kec.columns = [str(c).split("[")[0].strip() for c in df_kec.columns]
df_kec = df_kec[df_kec["Kode Kemendagri"].astype(str).str.match(r"^35\.78\.\d{2}$")].copy()
df_kec = df_kec.rename(columns={"Kode Kemendagri": "kode_kemendagri", "Kecamatan": "kecamatan",
                                "Kodepos": "kodepos", "Jumlah Kelurahan": "jumlah_kelurahan",
                                "Daftar Kelurahan": "daftar_kelurahan"})
df_kec["kecamatan"] = df_kec["kecamatan"].str.strip()
df_kec["jumlah_kelurahan"] = pd.to_numeric(df_kec["jumlah_kelurahan"], errors="coerce").astype("Int64")

print("Jumlah kecamatan:", len(df_kec), "| duplikat kode:", df_kec["kode_kemendagri"].duplicated().sum())
print("Total kelurahan :", int(df_kec["jumlah_kelurahan"].sum()))
df_kec.to_csv(PROCESSED_DIR / "kecamatan_surabaya.csv", index=False)
print("Disimpan: data/processed/kecamatan_surabaya.csv")
df_kec[["kode_kemendagri", "kecamatan", "kodepos", "jumlah_kelurahan"]].head(10)

### 4.3 Provenance dan atribusi lisensi

In [ ]:
# Bukti lisensi dari footer halaman Wikipedia, lalu catat provenance sumber (c)
sup_wiki = BeautifulSoup(isi_wiki, "lxml")
footer = sup_wiki.select_one("#footer-info-copyright")
print("Kutipan footer lisensi:", " ".join(footer.get_text(" ", strip=True).split()) if footer else "(tidak ditemukan)")
print("Tautan lisensi:", [a["href"] for a in footer.select("a[href*='creativecommons']")] if footer else [])

ATRIBUSI_WIKI = ("Kontributor Wikipedia, \"Daftar kecamatan dan kelurahan di Kota Surabaya\", Wikipedia bahasa Indonesia, "
                 f"diakses {akses_wiki[:10]}, {WIKI_URL}. Lisensi CC BY-SA 4.0.")
print("\nAtribusi:", ATRIBUSI_WIKI)
catat_provenance("C_wikipedia_html", "scraping (HTML)", WIKI_URL, WIKI_URL, "CC BY-SA 4.0",
                 PATH_WIKI_HTML, akses_wiki, "scraping lewat etis_get setelah can_fetch()=True; parse pd.read_html",
                 "tabel kecamatan hanya dipakai untuk Soal 4 (tidak di-merge)")
catat_provenance("C_robots", "bukti (robots.txt)", "https://id.wikipedia.org/robots.txt",
                 "https://id.wikipedia.org/robots.txt", "-", RAW_DIR / "wikipedia_id_robots.txt",
                 akses_robots_wiki, "diunduh lewat etis_get")

# Bukti status code request scraping (kosong jika HTML mentah sudah ada sebelumnya)
log_wiki = [r for r in LOG_REQUEST if r["host"] == "id.wikipedia.org"]
display(pd.DataFrame(log_wiki)) if log_wiki else print("Tidak ada request Wikipedia baru (memakai HTML mentah yang sudah ada).")

## 5. Soal 5 — Penggabungan Data (Merge)

**Rancangan:** data kunjungan (sumber a, granular destinasi × bulan) digabung dengan cuaca bulanan Surabaya (sumber b, satu baris per bulan) pada kunci **`bulan`**, dengan `validate="many_to_one"`. Tabel kecamatan Wikipedia (sumber c) hanya dipakai untuk Soal 4 karena data kunjungan tidak memiliki kolom kecamatan.

### 5.1 Normalisasi kunci dan dokumentasi perubahan

In [ ]:
# Normalisasi kunci di kedua sisi; setiap perubahan dicatat di tabel dokumentasi
df_a = df_kunjungan.copy()
nama_sebelum = df_a["nama_destinasi"].copy()
df_a["nama_destinasi"] = df_a["nama_destinasi"].str.strip()                  # perubahan 1
df_a["bulan"] = df_a["periode"].map(no_bulan).astype("int64")                # perubahan 2
assert df_a["bulan"].notna().all(), "Ada nama bulan yang tidak dikenali"
df_a = df_a.rename(columns={"periode": "nama_bulan"})

perubahan_kunci = pd.DataFrame([
    ("kunjungan", "nama_destinasi", "str.strip() menghapus spasi di ujung",
     int((nama_sebelum != df_a["nama_destinasi"]).sum()), "'ALUN ALUN SURABAYA BALAI PEMUDA ' -> tanpa spasi akhir"),
    ("kunjungan", "periode -> bulan", "nama bulan Indonesia dipetakan ke angka 1-12",
     len(df_a), "'Januari' -> 1, ..., 'Desember' -> 12"),
    ("kunjungan", "tahun", f"difilter ke TAHUN_DATA = {TAHUN_DATA}",
     len(df_kunjungan_raw) - len(df_kunjungan), "baris 2026 dikecualikan dari analisis (tetap ada di raw)"),
    ("cuaca", "time -> bulan", "tanggal harian diagregasi per bulan (dt.month)",
     len(df_cuaca_harian), f"{len(df_cuaca_harian)} baris harian -> {len(df_cuaca_bulanan)} baris bulanan"),
], columns=["sumber", "kolom", "perubahan", "baris_terdampak", "contoh"])
perubahan_kunci

In [ ]:
# Pemeriksaan sebelum merge: kunci unik di sisi 'one' dan tidak ada duplikasi di sisi 'many'
KUNCI = ["bulan"]
print("Baris kunjungan (kiri) :", len(df_a))
print("Baris cuaca (kanan)    :", len(df_cuaca_bulanan))
print("Duplikat kunci di cuaca bulanan               :", df_cuaca_bulanan.duplicated(KUNCI).sum())
print("Duplikat (nama_destinasi, bulan) di kunjungan :", df_a.duplicated(["nama_destinasi", "bulan"]).sum())
print("Duplikat baris penuh di kunjungan             :", df_a.duplicated().sum())
print("Himpunan bulan kiri == kanan:", sorted(df_a["bulan"].unique()) == sorted(df_cuaca_bulanan["bulan"]))

### 5.2 Merge dengan validasi dan `indicator=True`

In [ ]:
# Merge many_to_one: banyak destinasi per bulan -> satu baris cuaca per bulan
gabungan = pd.merge(df_a, df_cuaca_bulanan, on=KUNCI, how="left",
                    validate="many_to_one", indicator=True)

# Bukti tidak ada duplikasi dan semua baris cocok
print("Baris sebelum merge:", len(df_a), "| sesudah merge:", len(gabungan))
print("Status _merge:", gabungan["_merge"].value_counts().to_dict())
print("duplicated().sum() (baris penuh)            :", gabungan.drop(columns="_merge").duplicated().sum())
print("duplicated(['nama_destinasi','bulan']).sum():", gabungan.duplicated(["nama_destinasi", "bulan"]).sum())

assert len(gabungan) == len(df_a), "Jumlah baris berubah setelah merge"
assert (gabungan["_merge"] == "both").all(), "Ada baris yang tidak cocok"
assert gabungan.drop(columns="_merge").duplicated().sum() == 0
assert gabungan.duplicated(["nama_destinasi", "bulan"]).sum() == 0
print("\nSemua pemeriksaan lolos.")

In [ ]:
# Cek baris yang tidak cocok dari kedua arah dengan outer join + indicator
cek_outer = pd.merge(df_a[KUNCI].drop_duplicates(), df_cuaca_bulanan[KUNCI], on=KUNCI, how="outer", indicator=True)
print("Bulan hanya di kunjungan:", cek_outer.loc[cek_outer["_merge"] == "left_only", "bulan"].tolist())
print("Bulan hanya di cuaca    :", cek_outer.loc[cek_outer["_merge"] == "right_only", "bulan"].tolist())

In [ ]:
# Susun kolom akhir, simpan gabungan.csv, lalu tampilkan contoh baris
kolom_akhir = ["tahun", "bulan", "nama_bulan", "kode_kota", "nama_kota", "nama_destinasi", "wisnus", "wisman",
               "suhu_rata2_c", "suhu_maks_c", "suhu_min_c", "curah_hujan_mm", "jam_hujan", "hari_hujan",
               "sinar_matahari_jam", "angin_maks_kmh"]
gabungan_final = gabungan[kolom_akhir].sort_values(["bulan", "nama_destinasi"]).reset_index(drop=True)
PATH_GABUNGAN = PROCESSED_DIR / "gabungan.csv"
gabungan_final.to_csv(PATH_GABUNGAN, index=False, encoding="utf-8")
print("Disimpan:", PATH_GABUNGAN.relative_to(BASE_DIR), gabungan_final.shape)
gabungan_final.head(8)

In [ ]:
# Metadata hasil gabungan: sumber, ukuran, kunci, parameter validate, tanggal proses, MD5
metadata = {
    "judul": "Kunjungan wisatawan per destinasi x cuaca bulanan Kota Surabaya",
    "tahun_data": TAHUN_DATA,
    "sumber": [
        {"id": "A_csv_kunjungan", "nama": "Banyaknya Kunjungan Wisatawan Nusantara dan Wisatawan Mancanegara menurut Tempat Wisata",
         "penerbit": "Disbudporapar Kota Surabaya (Open Data Surabaya)", "url": CKAN_CSV_URL,
         "lisensi": "tidak dicantumkan di metadata CKAN", "baris_dipakai": int(len(df_a))},
        {"id": "B_open_meteo", "nama": "Open-Meteo Historical Weather API (harian -> bulanan)", "url": OPEN_METEO_URL,
         "parameter": params_cuaca, "lisensi": "CC BY 4.0", "baris_dipakai": int(len(df_cuaca_bulanan))},
    ],
    "sumber_tidak_digabung": [{"id": "C_wikipedia_html", "url": WIKI_URL, "alasan": "tidak ada kolom kecamatan di data kunjungan; hanya Soal 4"}],
    "kunci_merge": KUNCI,
    "perubahan_kunci": perubahan_kunci.to_dict(orient="records"),
    "how": "left",
    "validate": "many_to_one",
    "baris_sebelum_merge": int(len(df_a)),
    "baris_sesudah_merge": int(len(gabungan_final)),
    "jumlah_kolom": int(gabungan_final.shape[1]),
    "kolom": list(gabungan_final.columns),
    "status_merge": {k: int(v) for k, v in gabungan["_merge"].value_counts().items()},
    "duplikat": int(gabungan_final.duplicated().sum()),
    "catatan_kualitas": [f"Angka {a} dan {b} {TAHUN_DATA} identik untuk semua destinasi (dugaan kesalahan salin di sumber; tidak dikoreksi)"
                         for a, b in BULAN_IDENTIK],
    "tanggal_proses": datetime.now(WIB).isoformat(timespec="seconds"),
    "berkas": str(PATH_GABUNGAN.relative_to(BASE_DIR)),
    "md5": md5_file(PATH_GABUNGAN),
}
with open(PROCESSED_DIR / "gabungan_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps({k: metadata[k] for k in ["kunci_merge", "validate", "baris_sebelum_merge", "baris_sesudah_merge",
                                           "jumlah_kolom", "status_merge", "duplikat", "tanggal_proses", "md5"]},
                 ensure_ascii=False, indent=2))

### 5.3 Presentasi singkat hasil gabungan
Agregasi per bulan (jumlah wisnus + wisman seluruh destinasi) bersanding dengan curah hujan dan suhu, untuk melihat pola musiman.

In [ ]:
# Ringkasan bulanan: total kunjungan semua destinasi vs. variabel cuaca
ringkas_bulanan = (gabungan_final.groupby(["bulan", "nama_bulan"], sort=True)
                   .agg(wisnus=("wisnus", "sum"), wisman=("wisman", "sum"),
                        curah_hujan_mm=("curah_hujan_mm", "first"), hari_hujan=("hari_hujan", "first"),
                        suhu_rata2_c=("suhu_rata2_c", "first"))
                   .reset_index())
ringkas_bulanan["total_kunjungan"] = ringkas_bulanan["wisnus"] + ringkas_bulanan["wisman"]
display(ringkas_bulanan)

# Peringatan kualitas data ikut ditampilkan agar korelasi tidak dibaca berlebihan
if BULAN_IDENTIK:
    print("Perhatian: bulan identik", BULAN_IDENTIK, "- korelasi di bawah ikut terpengaruh.\n")

# Korelasi Spearman = korelasi Pearson atas peringkat (tanpa scipy); 12 titik, indikatif, bukan sebab-akibat
peringkat = ringkas_bulanan[["total_kunjungan", "curah_hujan_mm", "suhu_rata2_c"]].rank()
print("Korelasi Spearman total_kunjungan vs curah_hujan_mm:",
      round(peringkat["total_kunjungan"].corr(peringkat["curah_hujan_mm"]), 3))
print("Korelasi Spearman total_kunjungan vs suhu_rata2_c  :",
      round(peringkat["total_kunjungan"].corr(peringkat["suhu_rata2_c"]), 3))

In [ ]:
# Grafik: total kunjungan (batang) dan curah hujan bulanan (garis) dalam satu tampilan
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(ringkas_bulanan["nama_bulan"].str[:3], ringkas_bulanan["total_kunjungan"], color="#4C78A8")
ax1.set_ylabel("Total kunjungan (orang)")
ax1.set_title(f"Kunjungan 16 destinasi vs curah hujan — Kota Surabaya {TAHUN_DATA}")
ax2 = ax1.twinx()
ax2.plot(ringkas_bulanan["nama_bulan"].str[:3], ringkas_bulanan["curah_hujan_mm"], color="#E45756", marker="o")
ax2.set_ylabel("Curah hujan (mm/bulan)")
fig.tight_layout()
plt.show()

## 6. Ringkasan Akhir

### 6.1 Tabel provenance semua sumber

In [ ]:
# Simpan dan tampilkan provenance semua sumber
df_prov = pd.DataFrame(PROVENANCE)
df_prov.to_csv(BASE_DIR / "provenance.csv", index=False, encoding="utf-8")
print("Disimpan: provenance.csv |", len(df_prov), "baris")
with pd.option_context("display.max_colwidth", 70):
    display(df_prov)

In [ ]:
# Verifikasi integritas: MD5 setiap berkas raw dihitung ulang dan dibandingkan dengan provenance
verifikasi = df_prov[["sumber_id", "path_lokal", "md5"]].copy()
verifikasi["md5_sekarang"] = [md5_file(BASE_DIR / p) for p in verifikasi["path_lokal"]]
verifikasi["tidak_berubah"] = verifikasi["md5"] == verifikasi["md5_sekarang"]
assert verifikasi["tidak_berubah"].all(), "Ada berkas raw yang berubah!"
display(verifikasi)
print("Semua berkas di data/raw tidak berubah sejak dicatat.")

In [ ]:
# Log request HTTP: bukti timeout/jeda/status code untuk setiap akses web
df_log = pd.DataFrame(LOG_REQUEST)
if df_log.empty:
    print("Tidak ada request baru pada eksekusi ini (semua berkas mentah sudah ada).")
else:
    display(df_log)

In [ ]:
# Daftar berkas keluaran beserta ukurannya
keluaran = sorted([p for p in BASE_DIR.rglob("*") if p.is_file()
                   and (p.parts[len(BASE_DIR.parts)] == "data" or p.name in
                        {"provenance.csv", "data_ethics_checklist.md"})])
pd.DataFrame({"berkas": [str(p.relative_to(BASE_DIR)) for p in keluaran],
              "ukuran_byte": [p.stat().st_size for p in keluaran]})

### 6.2 Alur akuisisi (bahan demo)
1. **Setup etis:** email kontak dibaca dari `.env`, User-Agent jujur ditampilkan; `etis_get()` menerapkan timeout, jeda per host, retry + *exponential backoff* untuk 429/5xx, dan mencatat status code.
2. **Berkas (Soal 1):** robots.txt Open Data Surabaya diperiksa (API dilarang, `Crawl-delay: 10`), lalu CSV kunjungan diunduh ke `data/raw/`; parameter `read_csv` dipilih setelah memeriksa byte mentah; `TAHUN_DATA` diambil dari tahun yang lengkap 12 bulan.
3. **Etika (Soal 2):** lisensi dicatat apa adanya (tidak dicantumkan), dasar hukum portal dikutip, tidak ada PII/quasi-identifier, checklist 9 butir disimpan.
4. **API (Soal 3):** satu permintaan Open-Meteo untuk satu titik Kota Surabaya pada rentang bulan yang sama dengan data kunjungan; JSON mentah disimpan; dinormalisasi menjadi tabel harian (≥30 baris) lalu bulanan.
5. **Scraping (Soal 4):** robots.txt Wikipedia dibaca, `can_fetch()` = True, HTML disimpan mentah, tabel kecamatan diambil dengan `pd.read_html`, atribusi CC BY-SA dicatat.
6. **Merge (Soal 5):** kunci `bulan` dinormalisasi dan perubahannya didokumentasikan; `validate="many_to_one"` dan `indicator=True` membuktikan jumlah baris tetap dan semua baris `both`; hasil plus metadata dan MD5 disimpan.
7. **Provenance & integritas:** `provenance.csv` mencatat URL, tanggal akses, lisensi, MD5 setiap sumber; MD5 raw dihitung ulang untuk membuktikan data mentah tidak berubah.

**Atribusi:** Data kunjungan © Disbudporapar Kota Surabaya via Open Data Surabaya (lisensi tidak dicantumkan). Data cuaca: Open-Meteo.com, CC BY 4.0. Tabel kecamatan: kontributor Wikipedia, CC BY-SA 4.0.